# Hardware-aware Pareto hyperparameter search

This notebook runs the reusable `cv_search` modules from a fresh Google Colab runtime. It does not duplicate the model or training implementation in notebook cells.

Recommended order: setup → hardware inspection → synthetic smoke test → runtime preview → small CIFAR study → inspect Pareto results → export or resume.

## 1. Select a runtime

In Colab choose **Runtime → Change runtime type → GPU** when an accelerator is desired. GPU type and availability vary by session, so the hardware cell below is the source of truth.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/TrueRottweiler/WashingtonCsed504.git"
REPO_DIR = Path("/content/WashingtonCsed504")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

## 2. Install the framework

Colab normally includes PyTorch. Editable installation adds Optuna, psutil, plotting, and notebook dependencies without copying project logic into this notebook.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    check=True,
)
print("Installation complete")

## 3. Inspect hardware and software

In [ ]:
import json
import torch
from cv_search.hardware import inspect_hardware

hardware = inspect_hardware(Path("/content/colab_hardware.json"))
print(json.dumps(hardware, indent=2, default=str))
print("Selected accelerator available:", "CUDA" if torch.cuda.is_available() else "CPU/MPS")

## 4. Reproducibility setup

The study configuration records trial, split, and sampler seeds. This cell establishes an explicit notebook seed for any exploratory code outside the engine.

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Notebook seed:", SEED)

## 5. Framework overview

The two launchers use one engine:

- proxy screening;
- checkpoint-continuing successive halving;
- from-scratch multi-seed confirmation;
- optional final test evaluation only after selection;
- persistent Optuna/SQLite study, JSONL audit log, CSV leaderboard, Pareto export, reports, and plots.

In [ ]:
from cv_search.registry import registered_datasets, registered_models
from cv_search import adapters as _adapters  # registers built-ins
from cv_search import data as _data          # registers built-ins

print("Model adapters:", registered_models())
print("Datasets:", registered_datasets())

## 6. Lightweight synthetic smoke tests

These test software behavior, not model quality. Their validation scores are random-data output and must not be reported as accuracy results.

In [ ]:
SMOKE_MODEL = "transformer"  # change to "cnn" when desired
smoke_script = (
    "src/a1-cv/search_transformer.py"
    if SMOKE_MODEL == "transformer"
    else "src/a1-cv/search_cnn.py"
)
smoke_config = (
    "configs/searches/vit_cifar10.toml"
    if SMOKE_MODEL == "transformer"
    else "configs/searches/resnet_cifar10.toml"
)
subprocess.run(
    [
        sys.executable,
        smoke_script,
        "--config",
        smoke_config,
        "--smoke-test",
        "--skip-calibration",
    ],
    check=True,
)

## 7. Create small Colab CIFAR-10 configurations

The repository defaults are research-scale. These starter studies keep the test set isolated and fit a first Colab validation run.

In [ ]:
from textwrap import dedent

COLAB_CONFIG_DIR = Path("configs/searches")
COLAB_CONFIG_DIR.mkdir(parents=True, exist_ok=True)


def write_colab_config(adapter: str) -> Path:
    name = "resnet-colab-starter" if adapter == "cnn" else "vit-colab-starter"
    path = COLAB_CONFIG_DIR / f"{name}.toml"
    path.write_text(dedent(f'''        [study]
        name = "{name}"
        mode = "continuous"
        output_dir = "results"

        [model]
        adapter = "{adapter}"
        profile = "simple"

        [dataset]
        name = "cifar10"
        root = "data"
        download = true
        validation_fraction = 0.1
        split_seed = 42
        augmentation = "basic"

        [search]
        sampler = "tpe"
        trials = 4
        seed = 42
        concurrency = 1

        [execution]
        device = "auto"
        precision = "auto"
        reproducibility = "balanced"
        compile = false
        channels_last = true

        [objectives.validation_accuracy]
        enabled = true
        direction = "maximize"
        weight = 1.0

        [objectives.wall_clock_seconds]
        enabled = true
        direction = "minimize"
        weight = 0.25

        [objectives.peak_memory_mb]
        enabled = true
        direction = "minimize"
        weight = 0.15

        [constraints]
        minimum_free_disk_gb = 1

        [selection]
        policy = "weighted_pareto"

        [stages.proxy]
        enabled = true
        trials = 4
        epochs = 1
        max_train_steps = 100
        max_validation_batches = 5
        data_fraction = 0.25
        top_k = 2

        [stages.halving]
        enabled = true
        resource = "epochs"
        budgets = [2, 4]
        reduction_factor = 2
        minimum_candidates = 1
        continue_checkpoints = true

        [stages.full]
        enabled = true
        top_k = 1
        epochs = 5
        seeds = [7]
        evaluate_test = false
    '''))
    return path

RESNET_COLAB_CONFIG = write_colab_config("cnn")
VIT_COLAB_CONFIG = write_colab_config("transformer")
print(RESNET_COLAB_CONFIG)
print(VIT_COLAB_CONFIG)

## 8. Calibrate and preview runtime

The preview runs measured pilot steps and returns ranges. It is an estimate, not a guarantee.

In [ ]:
MODEL = "transformer"  # "cnn" or "transformer"
script = "src/a1-cv/search_transformer.py" if MODEL == "transformer" else "src/a1-cv/search_cnn.py"
config_path = VIT_COLAB_CONFIG if MODEL == "transformer" else RESNET_COLAB_CONFIG

subprocess.run(
    [
        sys.executable,
        script,
        "--config",
        str(config_path),
        "--estimate-only",
        "--calibration-steps",
        "10",
    ],
    check=True,
)

## 9. Run the configurable search

Set `RUN_SEARCH = True` only after reviewing the preview. Re-running the same unchanged study name reuses its database and completed stage records.

In [ ]:
RUN_SEARCH = False

if RUN_SEARCH:
    subprocess.run(
        [
            sys.executable,
            script,
            "--config",
            str(config_path),
            "--calibration-steps",
            "10",
            "--resume",
        ],
        check=True,
    )
else:
    print("Search not started. Set RUN_SEARCH = True after reviewing the estimate.")

## 10. Optimized repository ResNet and ViT examples

The files below preserve strong repository baselines and clearly separate repository-adapted settings from original-paper settings:

```text
configs/experiments/resnet_reference.toml
configs/experiments/vit_reference.toml
```

Copy either file, give it a new study name, and adjust the budget rather than editing a completed study in place.

In [ ]:
print(Path("configs/experiments/resnet_reference.toml").read_text()[:2000])
print("\n--- ViT reference ---\n")
print(Path("configs/experiments/vit_reference.toml").read_text()[:2000])

## 11. Inspect leaderboard and Pareto front

In [ ]:
import pandas as pd

study_dir = Path("results") / ("vit-colab-starter" if MODEL == "transformer" else "resnet-colab-starter")
leaderboard_path = study_dir / "leaderboard.csv"
pareto_path = study_dir / "pareto.csv"

if leaderboard_path.exists():
    leaderboard = pd.read_csv(leaderboard_path)
    display(leaderboard)
else:
    print("Run the search first:", leaderboard_path)

if pareto_path.exists():
    pareto = pd.read_csv(pareto_path)
    display(pareto)
else:
    print("Pareto export not present yet:", pareto_path)

## 12. Time, data, memory, compute, and cost estimates

In [ ]:
estimate_path = study_dir / "runtime_estimate.json"
results_path = study_dir / "results.jsonl"

if estimate_path.exists():
    estimate = json.loads(estimate_path.read_text())
    print(json.dumps(estimate, indent=2))
else:
    print("No runtime estimate yet")

if results_path.exists():
    records = [json.loads(line) for line in results_path.read_text().splitlines() if line.strip()]
    measured = {
        "executions": len(records),
        "elapsed_seconds": sum(float(row["elapsed_seconds"]) for row in records),
        "examples_processed": sum(int(row["examples_processed"]) for row in records),
        "gpu_hours": sum(float(row["gpu_hours"]) for row in records),
        "cpu_hours": sum(float(row["cpu_hours"]) for row in records),
        "estimated_cost_usd": sum(float(row["estimated_cost_usd"]) for row in records),
    }
    print("Measured execution totals:")
    print(json.dumps(measured, indent=2))

## 13. Optional Google Drive persistence

For performance, run under `/content` and copy completed study directories to Drive. Preserve the database, JSONL records, and checkpoints together.

In [ ]:
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/WashingtonCsed504-results")
    destination.mkdir(parents=True, exist_ok=True)
    if study_dir.exists():
        subprocess.run(["cp", "-r", str(study_dir), str(destination)], check=True)
        print("Copied to", destination)
else:
    print("Set MOUNT_DRIVE = True to persist results.")

## 14. Resume a study

Restore the whole study directory to its configured `output_dir/name`, then run the same command with the same configuration and `--resume`. Use a new study name when dataset, split, preprocessing, metric, or architecture semantics change.

In [ ]:
RESUME = False
if RESUME:
    subprocess.run(
        [sys.executable, script, "--config", str(config_path), "--resume", "--skip-calibration"],
        check=True,
    )
else:
    print("Set RESUME = True after restoring the study directory.")

## 15. Export results

In [ ]:
EXPORT = False
if EXPORT and study_dir.exists():
    archive = Path("/content") / f"{study_dir.name}-results.zip"
    subprocess.run(["zip", "-qr", str(archive), str(study_dir)], check=True)
    from google.colab import files
    files.download(str(archive))
else:
    print("Set EXPORT = True after a study completes.")

## 16. Scaling guidance

1. Confirm both smoke tests.
2. Run a 10–25 step calibration.
3. Start with 2–4 proxy trials and one confirmation seed.
4. Increase proxy trials before widening architecture ranges.
5. Add halving rungs after throughput is known.
6. Increase confirmation epochs, then seed count.
7. Keep test evaluation off until the final configuration is selected.
8. Do not interpret proxy or random-data scores as final evidence.

See `docs/COLAB.md`, `docs/CONFIGURATION.md`, and `docs/LIMITATIONS.md` for details.

## Multi-GPU execution

The framework automatically falls back to serial execution when only one accelerator is visible. On a multi-GPU runtime, use one independent trial per GPU during proxy/halving and optional DDP for full confirmation. Colab commonly exposes a single GPU, so check the count before selecting the multi-GPU configuration.


In [ ]:
import torch

gpu_count = torch.cuda.device_count()
print("Visible CUDA GPUs:", gpu_count)
for index in range(gpu_count):
    print(index, torch.cuda.get_device_name(index))


In [ ]:
# Run only on a runtime with at least two visible GPUs.
if gpu_count >= 2:
    !python src/a1-cv/search_cnn.py \
        --config configs/searches/resnet_cifar10_multi_gpu.toml \
        --gpu-indices 0,1
else:
    print("Multi-GPU example skipped; use the standard single-GPU configuration.")
